In [10]:
import pandas as pd

In [11]:
# Cleaned data
energy_df = pd.read_excel(r'data\MetalliCan\pre_cleaned_data\energy_df.xlsx')
material_df = pd.read_excel(r'data\MetalliCan\pre_cleaned_data\material_df.xlsx')
biosphere_df = pd.read_excel(r'data\MetalliCan\pre_cleaned_data\biosphere_df.xlsx')
land_df = pd.read_excel(r'data\MetalliCan\pre_cleaned_data\land_df.xlsx')

In [12]:
# Prices and production data
price_df = pd.read_excel(r'data/Prices/Prices_data.xlsx', sheet_name='data')
production_df = pd.read_excel(r'data/MetalliCan/sites_for_lci.xlsx', sheet_name='prod_data')

In [13]:
from utils.data_manipulations import build_activity_name, add_site_id

In [14]:
production_df

,main_id,facility_group_id,facility_name,facility_group_name,province,facility_type,mining_processing_type,npv,archetypes,biosphere_data?,...,Ni_conc,Mo_conc,Zn_conc,Pb_conc,Fe_conc,Pt_conc,Pd_conc,U_conc,Nb_conc,Au_conc
0,AB-MAIN-d3a4aba9,NaN,The Cobalt Refinery Company Inc.,NaN,Alberta,manufacturing,Refinery,Cold Evergreen Needleleaf Forest,Co hydrometallurgical refinery,NPRI+GHG,...,201675.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
1,BC-MAIN-3bb6b7cd,NaN,Trail,NaN,British Columbia,manufacturing,Refinery,Cool Evergreen Needleleaf Forest,Zn refinery,NPRI+GHG+Water,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
2,BC-MAIN-3f490561,NaN,Mount Polley,NaN,British Columbia,mining,"Open-pit, concentrator",Cold Evergreen Needleleaf Forest,"Porphyry sulfide, flotation-based",NPRI+GHG+Land,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.000000
3,BC-MAIN-4724f4ba,NaN,Elk,NaN,British Columbia,mining,Open-pit,Cold Evergreen Needleleaf Forest,Free-milling,Land,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.124287
4,BC-MAIN-599152a0,NaN,Copper Mountain,NaN,British Columbia,mining,"Open-pit, concentrator",Cool Evergreen Needleleaf Forest,"Porphyry sulfide, flotation-based",NPRI+GHG+Land,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,NaN,GRP-0d911886,NaN,Porcupine complex,Ontario,mining,"Open-pit, underground",NaN,Free-milling,NPRI+GHG+Land,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,8.251949
62,NaN,GRP-147b3123,NaN,Timmins Operation,Ontario,mining,"Underground, concentrator",NaN,Free-milling,NPRI+GHG+Land,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,4.218015
63,NaN,GRP-14bfbb82,NaN,Seabee Gold Operation,Saskatchewan,mining,"Underground, concentrator",NaN,Free-milling,NPRI+GHG+Land,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,1.230080
64,NaN,GRP-a13779f8,NaN,Snow Lake,Manitoba,mining,"Underground, concentrator",NaN,Free-milling,NPRI+Land,...,0.0,0.0,69284.0,0.0,0.0,NaN,NaN,NaN,0.0,32.978787


In [15]:
# Add site_id to dataframes
production_df = add_site_id(production_df)
energy_df = add_site_id(energy_df)
material_df = add_site_id(material_df)
biosphere_df = add_site_id(biosphere_df)
land_df = add_site_id(land_df)

In [16]:
# Add activitiy_name to production_df
production_df['activity_name'] = production_df.apply(lambda row: build_activity_name(row, production_df), axis=1)

In [17]:
# Add NPV to land df
land_df = land_df.merge(production_df[['site_id', 'npv']], on='site_id', how='left')

In [19]:
energy_df = energy_df.merge(production_df[['site_id', 'activity_name', 'mining_processing_type', 'archetypes']], on='site_id', how='left')
material_df = material_df.merge(production_df[['site_id', 'activity_name', 'mining_processing_type', 'archetypes']], on='site_id', how='left')
biosphere_df = biosphere_df.merge(production_df[['site_id', 'activity_name', 'mining_processing_type', 'archetypes']], on='site_id', how='left')
land_df = land_df.merge(production_df[['site_id', 'activity_name','archetypes']], on='site_id', how='left')

In [20]:
# To avoid double counting
biosphere_df = biosphere_df[~((biosphere_df['source_id'] == 'https://www.canada.ca/en/environment-climate-change/services/environmental-indicators/greenhouse-gas-emissions/large-facilities.html; https://open.canada.ca/data/en/dataset/a8ba14b7-7f23-462a-bdbb-83b0ef629823') & (biosphere_df['substance_id'] == 'NA - GHG'))]

In [21]:
substance_ghg = ['124-38-9']
release_pathway = ['Stationary Fuel Combustion', 'On-site Transportation']
CO2_df = biosphere_df[
    biosphere_df['substance_id'].isin(substance_ghg) &
    biosphere_df['release_pathway'].isin(release_pathway)
]

# Keep only relevant columns

In [22]:
energy_col = ['site_id', 'activity_name', 'mining_processing_type', 'archetypes', 'flow_type', 'subflow_type', 'value_MJ']
material_col = ['site_id', 'activity_name', 'mining_processing_type', 'archetypes', 'flow_type', 'subflow_type', 'mass_t']
biosphere_col = ['site_id', 'activity_name', 'mining_processing_type', 'archetypes', 'compartment_name', 'substance_name', 'flow_direction', 'release_pathway', 'unit', 'value']
land_col = ['site_id', 'activity_name', 'mining_processing_type', 'archetypes', 'area_m2', 'operation_periods']

In [23]:
energy_df = energy_df[energy_col]
material_df = material_df[material_col]
biosphere_df = biosphere_df[biosphere_col]
CO2_df = CO2_df[biosphere_col]
land_df = land_df[land_col]

In [24]:
# We create a tailings_df to add the quantity
tailings_df = production_df[['site_id', 'activity_name', 'mining_processing_type', 'archetypes', 'tailings_t_mass']]

In [25]:
tailings_df['flow_type'] = 'Material use'
tailings_df['subflow_type'] = 'Tailings'
tailings_df.rename(columns={'tailings_t_mass': 'value'}, inplace=True)

C:\Users\mp_ma\AppData\Local\Temp\ipykernel_17256\852003429.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tailings_df['flow_type'] = 'Material use'
C:\Users\mp_ma\AppData\Local\Temp\ipykernel_17256\852003429.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tailings_df['subflow_type'] = 'Tailings'
C:\Users\mp_ma\AppData\Local\Temp\ipykernel_17256\852003429.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://p

In [26]:
# Maybe need to differentiate unit and value in energy_df and material_df ?
energy_df['unit'] = 'MJ'
material_df['unit'] = 't'
tailings_df['unit'] = 't'
land_df['unit'] = 'm2'
energy_df.rename(columns={'value_MJ': 'value'}, inplace=True) # maybe a problem for normalization function
material_df.rename(columns={'mass_t': 'value'}, inplace=True) # maybe a problem for normalization function
land_df.rename(columns={'area_m2': 'value'}, inplace=True) # maybe a problem for normalization function

C:\Users\mp_ma\AppData\Local\Temp\ipykernel_17256\547898192.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tailings_df['unit'] = 't'


In [27]:
# To know the % inferred vs 'primary'
energy_df['data_source'] = 'MetalliCan'
material_df['data_source'] = 'MetalliCan'
biosphere_df['data_source'] = 'MetalliCan'
CO2_df['data_source'] = 'MetalliCan'
land_df['data_source'] = 'MetalliCan'

In [28]:
# To know the % inferred vs 'primary'
energy_df['value_formula'] = 'MetalliCan'
material_df['value_formula'] = 'MetalliCan'
biosphere_df['value_formula'] = 'MetalliCan'
CO2_df['value_formula'] = 'MetalliCan'
land_df['value_formula'] = 'MetalliCan'

# Data-gap filling

In [29]:
from core.data_gap_filling import *

In [30]:
# Initialize the InferenceEngine Class
engine = InferenceEngine(
    site_df=production_df,
    production_df=production_df,
    energy_df=energy_df,
    co2_df=CO2_df,
    land_df=land_df,
    material_df=material_df,
)

## Energy

In [21]:
site_id_to_fill_nrj = production_df[production_df['infer_energy_data'] == 'Yes']['site_id'].tolist()

In [22]:
ef = {
    "diesel": 2681,
    "natural_gas": 2354,
    "lpg": 2753
}

stationary_share_rules = {
    "Open-pit, concentrator": {"diesel": 0.7, "natural_gas": 0.2, "lpg": 0.1},
    "Underground, concentrator": {"diesel": 0.3, "natural_gas": 0.6, "lpg": 0.1},
}

default_shares = {"diesel": 0.3, "natural_gas": 0.6, "lpg": 0.1}

In [23]:
combined_energy_df, inferred_energy_df = engine.infer_energy_for_sites(
    site_ids=site_id_to_fill_nrj,
    ef_co2_per_unit=ef,
    stationary_share_rules=stationary_share_rules,
    default_shares=default_shares
)


In [24]:
inferred_energy_df

,site_id,activity_name,mining_processing_type,archetypes,flow_type,subflow_type,value,unit,data_source,value_formula
0,AB-MAIN-d3a4aba9,Refining at The Cobalt Refinery Company Inc.,Refinery,Co hydrometallurgical refinery,Energy,Diesel|Transport,6.613078e+05,L,Inference from CO2 (transport diesel),(1772.9661 * 1e6) / 2681
1,AB-MAIN-d3a4aba9,Refining at The Cobalt Refinery Company Inc.,Refinery,Co hydrometallurgical refinery,Energy,Diesel|Stationary,2.361087e+07,L,Inference from CO2 (stationary diesel),(63300.737969999995 * 1e6) / 2681
2,AB-MAIN-d3a4aba9,Refining at The Cobalt Refinery Company Inc.,Refinery,Co hydrometallurgical refinery,Energy,Natural Gas|Stationary,5.378143e+07,m3,Inference from CO2 (stationary natural_gas),(126601.47593999999 * 1e6) / 2354
3,AB-MAIN-d3a4aba9,Refining at The Cobalt Refinery Company Inc.,Refinery,Co hydrometallurgical refinery,Energy,Lpg|Stationary,7.664455e+06,L,Inference from CO2 (stationary lpg),(21100.24599 * 1e6) / 2753
4,NU-MAIN-5154702a,Open-pit mining at Mary River,Open-pit,Direct Shipping Ore,Energy,Diesel|Transport,2.419379e+07,L,Inference from CO2 (transport diesel),(64863.55484 * 1e6) / 2681
5,NU-MAIN-5154702a,Open-pit mining at Mary River,Open-pit,Direct Shipping Ore,Energy,Diesel|Stationary,2.649835e+06,L,Inference from CO2 (stationary diesel),(7104.207434999999 * 1e6) / 2681
6,NU-MAIN-5154702a,Open-pit mining at Mary River,Open-pit,Direct Shipping Ore,Energy,Natural Gas|Stationary,6.035860e+06,m3,Inference from CO2 (stationary natural_gas),(14208.414869999999 * 1e6) / 2354
7,NU-MAIN-5154702a,Open-pit mining at Mary River,Open-pit,Direct Shipping Ore,Energy,Lpg|Stationary,8.601777e+05,L,Inference from CO2 (stationary lpg),(2368.069145 * 1e6) / 2753
8,SK-MAIN-60ba74c4,and beneficiation at McClean Lake,Concentrator,High-grade U acid-leach mills,Energy,Diesel|Transport,6.640070e+05,L,Inference from CO2 (transport diesel),(1780.2028 * 1e6) / 2681
9,SK-MAIN-60ba74c4,and beneficiation at McClean Lake,Concentrator,High-grade U acid-leach mills,Energy,Diesel|Stationary,2.138831e+06,L,Inference from CO2 (stationary diesel),(5734.206149999999 * 1e6) / 2681


## Material

In [25]:
site_id_to_fill_material = production_df[production_df['infer_material_data'] == 'Yes']['site_id'].tolist()

In [26]:
material_archetype_rules_df = pd.read_excel(r'data/SI/SI_LCIs_review.xlsx', sheet_name='Mat_inference')

In [27]:
engine.init_material_inference(material_archetype_rules_df)

In [28]:
combined_material_df, inferred_material_df = engine.infer_material_for_sites(
    site_ids=site_id_to_fill_material,
    overwrite=False,
)

⚠️ No material rules for archetype 'Co hydrometallurgical refinery'
⚠️ No material rules for archetype 'Cu smelter'
⚠️ No material rules for archetype 'Cu-Ni smelter and refinery'
⚠️ No material rules for archetype 'Fe concentrator'
⚠️ No material rules for archetype 'Fe concentrator + pellet plant'
⚠️ No material rules for archetype 'Hybrid free-milling-refactory'
⚠️ No material rules for archetype 'Hybrid free-milling-refactory'
⚠️ No material rules for archetype 'Hybrid free-milling-refactory'
⚠️ No material rules for archetype 'Hybrid free-milling-refactory'
⚠️ No material rules for archetype 'Nb mine'
⚠️ No material rules for archetype 'Nb smelter'
⚠️ No material rules for archetype 'Ni hydrometallurgical refinery'
⚠️ No material rules for archetype 'Ni-Cu smelter'
⚠️ No material rules for archetype 'Zn refinery'
⚠️ No material rules for archetype 'Zn refinery'


C:\Users\mp_ma\OneDrive - polymtl\POST_DOC\CODE\regionalized_lci_mineral\core\data_gap_filling.py:97: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([self.material_df, inferred], ignore_index=True)


In [29]:
inferred_material_df

,site_id,activity_name,mining_processing_type,archetypes,flow_type,subflow_type,value,unit,data_source,value_formula,amount_parameter,parameter_distribution,value_min,value_mean,value_max
0,NU-MAIN-5154702a,Open-pit mining at Mary River,Open-pit,Direct Shipping Ore,Material,Explosives,None,kg,Archetype inference (Drilling & Blasting) | Ir...,0.1-0.3 * ore_processed_t,None,"Uniform(0.1, 0.3)",560000.0,1120000.00,1680000.0
1,BC-MAIN-4724f4ba,Open-pit mining at Elk,Open-pit,Free-milling,Material,Sodium cyanide,None,kg,Archetype inference (Leaching (CIP/CIL)) | Nor...,1.0-1.5 * ore_processed_t,None,"Uniform(1.0, 1.5)",33245.0,41556.25,49867.5
2,BC-MAIN-4724f4ba,Open-pit mining at Elk,Open-pit,Free-milling,Material,Lime,None,kg,Archetype inference (Leaching (CIP/CIL)) | Ope...,0.5-2.0 * ore_processed_t,None,"Uniform(0.5, 2.0)",16622.5,41556.25,66490.0
3,BC-MAIN-4724f4ba,Open-pit mining at Elk,Open-pit,Free-milling,Material,Activated carbon,None,g,Archetype inference (Adsorption & Recovery) | ...,15-50 * ore_processed_t,None,"Uniform(15.0, 50.0)",498675.0,1080462.50,1662250.0
4,BC-MAIN-4724f4ba,Open-pit mining at Elk,Open-pit,Free-milling,Material,Flocculant,None,g,Archetype inference (Water treatment / Tailing...,18537 * ore_processed_t,None,None,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142,QC-MAIN-a97821c0,Underground mining and beneficiation at Raglan,"Underground, concentrator","Porphyry sulfide, flotation-based",Material,Lime,None,kg,Archetype inference (pH Control) | Rio Tinto C...,1.5-3.0 * ore_processed_t,None,"Uniform(1.5, 3.0)",2250000.0,3375000.00,4500000.0
143,QC-MAIN-e51eda66,Underground mining and beneficiation at LaRonde,"Underground, concentrator","Porphyry sulfide, flotation-based",Material,Collectors (xanthates),None,kg,Archetype inference (Flotation) | UNEP (1991);...,0.05-0.1 * ore_processed_t,None,"Uniform(0.05, 0.1)",75075.0,112612.50,150150.0
144,QC-MAIN-e51eda66,Underground mining and beneficiation at LaRonde,"Underground, concentrator","Porphyry sulfide, flotation-based",Material,Grinding media,None,kg,Archetype inference (Grinding) | Norgate & Haq...,0.5-1.0 * ore_processed_t,None,"Uniform(0.5, 1.0)",750750.0,1126125.00,1501500.0
145,QC-MAIN-e51eda66,Underground mining and beneficiation at LaRonde,"Underground, concentrator","Porphyry sulfide, flotation-based",Material,Frother (MIBC or Dowfroth),None,kg,Archetype inference (Flotation) | Flotation re...,0.01-0.03 * ore_processed_t,None,"Uniform(0.01, 0.03)",15015.0,30030.00,45045.0


## Cement

In [30]:
site_id_to_fill_material = production_df[production_df['infer_material_data'] == 'Yes']['site_id'].tolist()

In [31]:
cement_params = {
    "underground": {
        "cement_factor": (5, 50),
        "backfill_share": (0.3, 0.9),  # optional, future
    },
    "open_pit": None,
}


In [32]:
combined_cement_df, inferred_cement_df = engine.infer_cement_for_sites(
    site_ids=site_id_to_fill_material,
    cement_params=cement_params,
)

C:\Users\mp_ma\OneDrive - polymtl\POST_DOC\CODE\regionalized_lci_mineral\core\data_gap_filling.py:137: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.material_df = pd.concat(


In [33]:
inferred_cement_df

,site_id,activity_name,mining_processing_type,archetypes,flow_type,subflow_type,value,unit,data_source,value_formula,amount_parameter,parameter_distribution,value_min,value_mean,value_max
0,NU-MAIN-8b0264c9,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator",Free-milling,Material,Cement (backfilling),None,kg,Inference | Cement backfilling | underground |...,"ore_processed_t * Uniform(5, 50)",ore_processed_t,"Uniform(5, 50)",9.590715e+06,5.274893e+07,9.590715e+07
1,ON-MAIN-1f126a43,Underground mining and beneficiation at Macassa,"Underground, concentrator",Free-milling,Material,Cement (backfilling),None,kg,Inference | Cement backfilling | underground |...,"ore_processed_t * Uniform(5, 50)",ore_processed_t,"Uniform(5, 50)",2.207940e+06,1.214367e+07,2.207940e+07
2,ON-MAIN-4e0734b5,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator",Free-milling,Material,Cement (backfilling),None,kg,Inference | Cement backfilling | underground |...,"ore_processed_t * Uniform(5, 50)",ore_processed_t,"Uniform(5, 50)",2.285000e+06,1.256750e+07,2.285000e+07
3,ON-MAIN-6e9be24e,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator",Free-milling,Material,Cement (backfilling),None,kg,Inference | Cement backfilling | underground |...,"ore_processed_t * Uniform(5, 50)",ore_processed_t,"Uniform(5, 50)",6.330045e+06,3.481525e+07,6.330045e+07
4,ON-MAIN-f4fc3276,Underground mining and beneficiation at Sugar ...,"Underground, concentrator",Free-milling,Material,Cement (backfilling),None,kg,Inference | Cement backfilling | underground |...,"ore_processed_t * Uniform(5, 50)",ore_processed_t,"Uniform(5, 50)",1.173355e+06,6.453452e+06,1.173355e+07
5,QC-MAIN-c0660aec,Underground mining and beneficiation at Goldex,"Underground, concentrator",Free-milling,Material,Cement (backfilling),None,kg,Inference | Cement backfilling | underground |...,"ore_processed_t * Uniform(5, 50)",ore_processed_t,"Uniform(5, 50)",1.443464e+07,7.939050e+07,1.443464e+08
6,QC-MAIN-f9e41c2a,Underground mining and beneficiation at Lamaque,"Underground, concentrator",Free-milling,Material,Cement (backfilling),None,kg,Inference | Cement backfilling | underground |...,"ore_processed_t * Uniform(5, 50)",ore_processed_t,"Uniform(5, 50)",4.192095e+06,2.305652e+07,4.192095e+07
7,GRP-0a2c0d69,Open-pit and underground mining at Meadowbank ...,"Open-pit, underground",Free-milling,Material,Cement (backfilling),None,kg,Inference | Cement backfilling | underground |...,"ore_processed_t * Uniform(5, 50)",ore_processed_t,"Uniform(5, 50)",1.921325e+07,1.056728e+08,1.921325e+08
8,GRP-147b3123,Underground mining and beneficiation at Timmin...,"Underground, concentrator",Free-milling,Material,Cement (backfilling),None,kg,Inference | Cement backfilling | underground |...,"ore_processed_t * Uniform(5, 50)",ore_processed_t,"Uniform(5, 50)",7.870000e+06,4.328500e+07,7.870000e+07
9,GRP-14bfbb82,Underground mining and beneficiation at Seabee...,"Underground, concentrator",Free-milling,Material,Cement (backfilling),None,kg,Inference | Cement backfilling | underground |...,"ore_processed_t * Uniform(5, 50)",ore_processed_t,"Uniform(5, 50)",6.100000e+05,3.355000e+06,6.100000e+06


## Explosives

In [34]:
site_id_to_fill_explosives = production_df[production_df['infer_material_data'] == 'Yes']['site_id'].tolist()

In [35]:
explosives_params = {
    "open_pit": {
        "strip_ratio": (1.5, 6.0),       # t waste / t ore
        "explosive_factor": (0.25, 0.8), # kg explosives / t material
    },
    "underground": {
        "explosive_factor": (0.1, 0.3),  # kg explosives / t ore
    }
}

In [36]:
combined_explosives_df, inferred_explosives_df = engine.infer_explosives_for_sites(
    site_ids=site_id_to_fill_explosives,
    explosive_params=explosives_params,
)

C:\Users\mp_ma\OneDrive - polymtl\POST_DOC\CODE\regionalized_lci_mineral\core\data_gap_filling.py:118: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([self.material_df, inferred], ignore_index=True)


## Land

In [37]:
site_id_to_fill_land = production_df[production_df['infer_land_data'] == 'Yes']['site_id'].tolist()

In [38]:
combined_land_df, inferred_land_df = engine.infer_land_for_sites(
    site_ids=site_id_to_fill_land,
    formula_open_pit="0.791 * ore_processed_t - 7.76e5",
    formula_underground="Uniform(5e5, 2e6)",
    formula_other="Uniform(1e4, 1e5)",
    overwrite=False
)


C:\Users\mp_ma\OneDrive - polymtl\POST_DOC\CODE\regionalized_lci_mineral\core\data_gap_filling.py:77: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([self.land_df, inferred], ignore_index=True)


In [59]:
combined_land_df

,site_id,activity_name,mining_processing_type,archetypes,value,operation_periods,unit,data_source,value_formula,amount_parameter,parameter_distribution,value_min,value_mean,value_max
0,BC-MAIN-23155c25,NaN,Underground,NaN,1.499690e+06,1966–1985; 2002–2015; 2019–open,m2,MetalliCan,MetalliCan,NaN,NaN,NaN,NaN,NaN
1,BC-MAIN-3ef4f421,NaN,NaN,NaN,1.396089e+06,NaN,m2,MetalliCan,MetalliCan,NaN,NaN,NaN,NaN,NaN
2,BC-MAIN-3f490561,Open-pit mining and beneficiation at Mount Polley,"Open-pit, concentrator","Porphyry sulfide, flotation-based",7.967835e+06,NaN,m2,MetalliCan,MetalliCan,NaN,NaN,NaN,NaN,NaN
3,BC-MAIN-4724f4ba,Open-pit mining at Elk,Open-pit,Free-milling,4.167369e+05,NaN,m2,MetalliCan,MetalliCan,NaN,NaN,NaN,NaN,NaN
4,BC-MAIN-599152a0,Open-pit mining and beneficiation at Copper Mo...,"Open-pit, concentrator","Porphyry sulfide, flotation-based",1.323321e+07,1884–1958; 2011–open,m2,MetalliCan,MetalliCan,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132,ON-MAIN-687b8c8d,Underground mining and beneficiation at Island,"Underground, concentrator",Free-milling,NaN,None,m2,Inference (underground range),"Uniform(5e5, 2e6)",None,"Uniform(5e5, 2e6)",500000.000000,1.250000e+06,2.000000e+06
133,ON-MAIN-f4fc3276,Underground mining and beneficiation at Sugar ...,"Underground, concentrator",Free-milling,NaN,None,m2,Inference (underground range),"Uniform(5e5, 2e6)",None,"Uniform(5e5, 2e6)",500000.000000,1.250000e+06,2.000000e+06
134,BC-MAIN-857b7b89,Underground mining and beneficiation at Brucejack,"Underground, concentrator",Hybrid free-milling-refactory,NaN,None,m2,Inference (underground range),"Uniform(5e5, 2e6)",None,"Uniform(5e5, 2e6)",500000.000000,1.250000e+06,2.000000e+06
135,QC-MAIN-5ce331b8,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator","Porphyry sulfide, flotation-based",NaN,None,m2,Inference (open-pit model),(0.791 * 1082352.94117647) - 7.76e5,ore_processed_t,None,80141.176471,8.014118e+04,8.014118e+04


# Normalize flows

In [40]:
combined_material_df = pd.concat([combined_material_df, combined_cement_df, combined_explosives_df, tailings_df], ignore_index=True)

In [41]:
combined_energy_df

,site_id,activity_name,mining_processing_type,archetypes,flow_type,subflow_type,value,unit,data_source,value_formula
0,BC-MAIN-857b7b89,Underground mining and beneficiation at Brucejack,"Underground, concentrator",Hybrid free-milling-refactory,Energy,Acetylene,1.847565e+04,MJ,MetalliCan,MetalliCan
1,BC-MAIN-857b7b89,Underground mining and beneficiation at Brucejack,"Underground, concentrator",Hybrid free-milling-refactory,Energy,Aviation fuel,7.267611e+07,MJ,MetalliCan,MetalliCan
2,BC-MAIN-857b7b89,Underground mining and beneficiation at Brucejack,"Underground, concentrator",Hybrid free-milling-refactory,Energy,Diesel,2.870424e+08,MJ,MetalliCan,MetalliCan
3,BC-MAIN-857b7b89,Underground mining and beneficiation at Brucejack,"Underground, concentrator",Hybrid free-milling-refactory,Energy,Electricity consumption|Generated on-site,8.622512e+07,MJ,MetalliCan,MetalliCan
4,BC-MAIN-857b7b89,Underground mining and beneficiation at Brucejack,"Underground, concentrator",Hybrid free-milling-refactory,Energy,Electricity consumption|Grid electricity,5.690096e+08,MJ,MetalliCan,MetalliCan
...,...,...,...,...,...,...,...,...,...,...
211,QC-MAIN-a97821c0,Underground mining and beneficiation at Raglan,"Underground, concentrator","Porphyry sulfide, flotation-based",Energy,Lpg|Stationary,5.014061e+06,L,Inference from CO2 (stationary lpg),(13803.71125 * 1e6) / 2753
212,BC-MAIN-3bb6b7cd,Refining at Trail,Refinery,Zn refinery,Energy,Diesel|Transport,8.863841e+05,L,Inference from CO2 (transport diesel),(2376.3958 * 1e6) / 2681
213,BC-MAIN-3bb6b7cd,Refining at Trail,Refinery,Zn refinery,Energy,Diesel|Stationary,3.291620e+07,L,Inference from CO2 (stationary diesel),(88248.34095 * 1e6) / 2681
214,BC-MAIN-3bb6b7cd,Refining at Trail,Refinery,Zn refinery,Energy,Natural Gas|Stationary,7.497735e+07,m3,Inference from CO2 (stationary natural_gas),(176496.6819 * 1e6) / 2354


In [42]:
combined_material_df

,site_id,activity_name,mining_processing_type,archetypes,flow_type,subflow_type,value,unit,data_source,value_formula,amount_parameter,parameter_distribution,value_min,value_mean,value_max
0,QC-MAIN-b86f7d07,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator",Free-milling,Material use,Surface/underground emulsion & ANFO,2.968124e+03,t,MetalliCan,MetalliCan,NaN,NaN,NaN,NaN,NaN
1,ON-MAIN-aeafbb59,Open-pit mining and beneficiation at Detour Lake,"Open-pit, concentrator",Free-milling,Material use,Explosives,1.626850e+04,t,MetalliCan,MetalliCan,NaN,NaN,NaN,NaN,NaN
2,ON-MAIN-cb85213a,Underground mining and beneficiation at Eagle ...,"Underground, concentrator",Free-milling,Material use,Explosives,1.211000e+03,t,MetalliCan,MetalliCan,NaN,NaN,NaN,NaN,NaN
3,QC-MAIN-6dc537e6,Underground mining and beneficiation at Éléonore,"Underground, concentrator",Free-milling,Material use,Cement,2.737400e+04,t,MetalliCan,MetalliCan,NaN,NaN,NaN,NaN,NaN
4,QC-MAIN-6dc537e6,Underground mining and beneficiation at Éléonore,"Underground, concentrator",Free-milling,Material use,Grinding media,3.039900e+03,t,MetalliCan,MetalliCan,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
817,QC-MAIN-5ce331b8,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator","Porphyry sulfide, flotation-based",Material use,Tailings,7.192279e+05,t,NaN,NaN,NaN,NaN,NaN,NaN,NaN
818,QC-MAIN-a97821c0,Underground mining and beneficiation at Raglan,"Underground, concentrator","Porphyry sulfide, flotation-based",Material use,Tailings,1.348550e+06,t,NaN,NaN,NaN,NaN,NaN,NaN,NaN
819,QC-MAIN-e51eda66,Underground mining and beneficiation at LaRonde,"Underground, concentrator","Porphyry sulfide, flotation-based",Material use,Tailings,1.477492e+06,t,NaN,NaN,NaN,NaN,NaN,NaN,NaN
820,BC-MAIN-3bb6b7cd,Refining at Trail,Refinery,Zn refinery,Material use,Tailings,0.000000e+00,t,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [43]:
combined_material_df

,site_id,activity_name,mining_processing_type,archetypes,flow_type,subflow_type,value,unit,data_source,value_formula,amount_parameter,parameter_distribution,value_min,value_mean,value_max
0,QC-MAIN-b86f7d07,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator",Free-milling,Material use,Surface/underground emulsion & ANFO,2.968124e+03,t,MetalliCan,MetalliCan,NaN,NaN,NaN,NaN,NaN
1,ON-MAIN-aeafbb59,Open-pit mining and beneficiation at Detour Lake,"Open-pit, concentrator",Free-milling,Material use,Explosives,1.626850e+04,t,MetalliCan,MetalliCan,NaN,NaN,NaN,NaN,NaN
2,ON-MAIN-cb85213a,Underground mining and beneficiation at Eagle ...,"Underground, concentrator",Free-milling,Material use,Explosives,1.211000e+03,t,MetalliCan,MetalliCan,NaN,NaN,NaN,NaN,NaN
3,QC-MAIN-6dc537e6,Underground mining and beneficiation at Éléonore,"Underground, concentrator",Free-milling,Material use,Cement,2.737400e+04,t,MetalliCan,MetalliCan,NaN,NaN,NaN,NaN,NaN
4,QC-MAIN-6dc537e6,Underground mining and beneficiation at Éléonore,"Underground, concentrator",Free-milling,Material use,Grinding media,3.039900e+03,t,MetalliCan,MetalliCan,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
817,QC-MAIN-5ce331b8,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator","Porphyry sulfide, flotation-based",Material use,Tailings,7.192279e+05,t,NaN,NaN,NaN,NaN,NaN,NaN,NaN
818,QC-MAIN-a97821c0,Underground mining and beneficiation at Raglan,"Underground, concentrator","Porphyry sulfide, flotation-based",Material use,Tailings,1.348550e+06,t,NaN,NaN,NaN,NaN,NaN,NaN,NaN
819,QC-MAIN-e51eda66,Underground mining and beneficiation at LaRonde,"Underground, concentrator","Porphyry sulfide, flotation-based",Material use,Tailings,1.477492e+06,t,NaN,NaN,NaN,NaN,NaN,NaN,NaN
820,BC-MAIN-3bb6b7cd,Refining at Trail,Refinery,Zn refinery,Material use,Tailings,0.000000e+00,t,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [44]:
from core.normalization_allocation import normalize_flows

### Per ore processed

In [45]:
#energy_ore = normalize_flows(energy_df, production_df, mode='ore', value_col='value_MJ')
#material_ore = normalize_flows(material_df, production_df, mode='ore', value_col='mass')
#biosphere_ore = normalize_flows(biosphere_df, production_df, mode='ore', value_col='value')

In [52]:
combined_land_df

,site_id,activity_name,mining_processing_type,archetypes,value,operation_periods,unit,data_source,value_formula,amount_parameter,parameter_distribution,value_min,value_mean,value_max
0,BC-MAIN-23155c25,NaN,Underground,NaN,1.499690e+06,1966–1985; 2002–2015; 2019–open,m2,MetalliCan,MetalliCan,NaN,NaN,NaN,NaN,NaN
1,BC-MAIN-3ef4f421,NaN,NaN,NaN,1.396089e+06,NaN,m2,MetalliCan,MetalliCan,NaN,NaN,NaN,NaN,NaN
2,BC-MAIN-3f490561,Open-pit mining and beneficiation at Mount Polley,"Open-pit, concentrator","Porphyry sulfide, flotation-based",7.967835e+06,NaN,m2,MetalliCan,MetalliCan,NaN,NaN,NaN,NaN,NaN
3,BC-MAIN-4724f4ba,Open-pit mining at Elk,Open-pit,Free-milling,4.167369e+05,NaN,m2,MetalliCan,MetalliCan,NaN,NaN,NaN,NaN,NaN
4,BC-MAIN-599152a0,Open-pit mining and beneficiation at Copper Mo...,"Open-pit, concentrator","Porphyry sulfide, flotation-based",1.323321e+07,1884–1958; 2011–open,m2,MetalliCan,MetalliCan,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132,ON-MAIN-687b8c8d,Underground mining and beneficiation at Island,"Underground, concentrator",Free-milling,NaN,None,m2,Inference (underground range),"Uniform(5e5, 2e6)",None,"Uniform(5e5, 2e6)",500000.000000,1.250000e+06,2.000000e+06
133,ON-MAIN-f4fc3276,Underground mining and beneficiation at Sugar ...,"Underground, concentrator",Free-milling,NaN,None,m2,Inference (underground range),"Uniform(5e5, 2e6)",None,"Uniform(5e5, 2e6)",500000.000000,1.250000e+06,2.000000e+06
134,BC-MAIN-857b7b89,Underground mining and beneficiation at Brucejack,"Underground, concentrator",Hybrid free-milling-refactory,NaN,None,m2,Inference (underground range),"Uniform(5e5, 2e6)",None,"Uniform(5e5, 2e6)",500000.000000,1.250000e+06,2.000000e+06
135,QC-MAIN-5ce331b8,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator","Porphyry sulfide, flotation-based",NaN,None,m2,Inference (open-pit model),(0.791 * 1082352.94117647) - 7.76e5,ore_processed_t,None,80141.176471,8.014118e+04,8.014118e+04


### Per concentrate stream

In [48]:
energy_conc_econ = normalize_flows(combined_energy_df, production_df, price_df=price_df, mode='concentrate', allocation='economic', value_col='value')

In [50]:
material_conc_econ = normalize_flows(combined_material_df, production_df, price_df=price_df,  mode='concentrate', allocation='economic', value_col='value')

In [51]:
biosphere_conc_econ = normalize_flows(biosphere_df, production_df, price_df=price_df, mode='concentrate', allocation='economic', value_col='value')

In [57]:
land_conc_econ = normalize_flows(combined_land_df, production_df, price_df=price_df, mode='concentrate', allocation='economic', value_col='value')

### Per metal produced

In [21]:
#energy_metal_econ = normalize_flows(energy_df, production_df, price_df=price_df, mode='metal', allocation='economic', value_col='value_MJ')
#material_metal_econ = normalize_flows(material_df, production_df, price_df=price_df,  mode='metal', allocation='economic', value_col='mass')
#biosphere_metal_econ = normalize_flows(biosphere_df, production_df, price_df=price_df, mode='metal', allocation='economic', value_col='value')

# Keeping only relevant columns

In [58]:
land_conc_econ

,site_id,activity_name,mining_processing_type,archetypes,value,operation_periods,unit,data_source,value_formula,amount_parameter,...,concentrate,mass_conc,allocation_factor,facility_type,value_normalized,value_min_normalized,value_mean_normalized,value_max_normalized,functional_unit,normalization_key
0,BC-MAIN-3f490561,Open-pit mining and beneficiation at Mount Polley,"Open-pit, concentrator","Porphyry sulfide, flotation-based",7.967835e+06,NaN,m2,MetalliCan,MetalliCan,NaN,...,Cu,45578.436133,1.000000,mining,1.748159e+02,NaN,NaN,NaN,Cu concentrate,concentrate_economic
1,BC-MAIN-4724f4ba,Open-pit mining at Elk,Open-pit,Free-milling,4.167369e+05,NaN,m2,MetalliCan,MetalliCan,NaN,...,Au,0.124287,1.000000,mining,3.353019e+06,NaN,NaN,NaN,Doré,concentrate_economic
2,BC-MAIN-599152a0,Open-pit mining and beneficiation at Copper Mo...,"Open-pit, concentrator","Porphyry sulfide, flotation-based",1.323321e+07,1884–1958; 2011–open,m2,MetalliCan,MetalliCan,NaN,...,Cu,63500.000000,1.000000,mining,2.083970e+02,NaN,NaN,NaN,Cu concentrate,concentrate_economic
3,BC-MAIN-6b4800fe,Open-pit mining and beneficiation at Gibraltar,"Open-pit, concentrator","Porphyry sulfide, flotation-based",2.252800e+07,1972–1998; 2004–open,m2,MetalliCan,MetalliCan,NaN,...,Cu,185367.930667,0.999756,mining,1.215016e+02,NaN,NaN,NaN,Cu concentrate,concentrate_economic
4,BC-MAIN-6b4800fe,Open-pit mining and beneficiation at Gibraltar,"Open-pit, concentrator","Porphyry sulfide, flotation-based",2.252800e+07,1972–1998; 2004–open,m2,MetalliCan,MetalliCan,NaN,...,Mo,10.690541,0.000244,mining,5.148849e+02,NaN,NaN,NaN,Mo concentrate,concentrate_economic
5,BC-MAIN-8eb8be0d,Open-pit mining and beneficiation at Red Chris,"Open-pit, concentrator",Hybrid free-milling-refactory,1.153165e+07,2015–open,m2,MetalliCan,MetalliCan,NaN,...,Cu,12095.786667,1.000000,mining,9.533606e+02,NaN,NaN,NaN,Cu concentrate,concentrate_economic
6,BC-MAIN-aa76f6f2,Underground mining and beneficiation at New Afton,"Underground, concentrator","Porphyry sulfide, flotation-based",4.950460e+06,2012–open,m2,MetalliCan,MetalliCan,NaN,...,Cu,71667.536000,1.000000,mining,6.907535e+01,NaN,NaN,NaN,Cu concentrate,concentrate_economic
7,BC-MAIN-bf503b6b,Open-pit mining and beneficiation at Highland ...,"Open-pit, concentrator","Porphyry sulfide, flotation-based",6.392426e+07,NaN,m2,MetalliCan,MetalliCan,NaN,...,Cu,341333.333333,0.999781,mining,1.872371e+02,NaN,NaN,NaN,Cu concentrate,concentrate_economic
8,BC-MAIN-bf503b6b,Open-pit mining and beneficiation at Highland ...,"Open-pit, concentrator","Porphyry sulfide, flotation-based",6.392426e+07,NaN,m2,MetalliCan,MetalliCan,NaN,...,Mo,17.647059,0.000219,mining,7.934511e+02,NaN,NaN,NaN,Mo concentrate,concentrate_economic
9,BC-MAIN-ed23117f,Open-pit mining and beneficiation at Mount Mil...,"Open-pit, concentrator","Porphyry sulfide, flotation-based",1.260679e+07,2014–open,m2,MetalliCan,MetalliCan,NaN,...,Cu,93533.694347,1.000000,mining,1.347834e+02,NaN,NaN,NaN,Cu concentrate,concentrate_economic


In [24]:
energy_col = ['activity_name', 'functional_unit', 'site_id', 'subflow_type', 'unit', 'value_normalized']
material_col = ['activity_name', 'functional_unit', 'site_id', 'subflow_type', 'unit', 'value_normalized', 'value_min_normalized', 'value_mean_normalized', 'value_max_normalized']
biosphere_col = ['activity_name', 'functional_unit', 'site_id', 'substance_name', 'unit', 'value_normalized', 'value_min_normalized', 'value_mean_normalized', 'value_max_normalized']
land_col = ['activity_name', 'functional_unit', 'site_id', 'substance_name', 'unit', 'value_normalized', 'value_min_normalized', 'value_mean_normalized', 'value_max_normalized']

# Exports normalized dataframes

In [25]:
# energy_ore.to_csv(r'data/MetalliCan/data_for_lci_initialization/ore/energy_df.csv', index=False)
# material_ore.to_csv(r'data/MetalliCan/data_for_lci_initialization/ore/material_df.csv', index=False)
# biosphere_ore.to_csv(r'data/MetalliCan/data_for_lci_initialization/ore/biosphere_df.csv', index=False)

In [27]:
# energy_metal_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/metal/energy_df.csv', index=False)
# material_metal_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/metal/material_df.csv', index=False)
# biosphere_metal_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/metal/biosphere_df.csv', index=False)

In [ ]:
energy_conc_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/energy_df.csv', index=False)
material_conc_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/material_df.csv', index=False)
biosphere_conc_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/biosphere_df.csv', index=False)